# Mean-Reverting Processes & Interest Rate Models

From-scratch implementations of the Ornstein-Uhlenbeck process, Vasicek model, CIR model, and yield curve fitting.

**Outline**
1. Why mean reversion? The rubber band analogy
2. The Ornstein-Uhlenbeck process
3. Exact solution and simulation
4. Stationary distribution
5. The Vasicek model
6. Bond pricing under Vasicek
7. The CIR model
8. CIR bond pricing
9. Yield curve shapes
10. Parameter calibration via MLE
11. Model comparison
12. Summary
13. References

### Why this notebook matters

Mean-reverting processes are everywhere in finance:
* **Interest rates** revert to long-run average levels
* **Volatility** spikes during crises and decays back to baseline
* **Pairs trading** exploits short-term deviations from long-run equilibrium spreads
* **Commodity prices** revert toward marginal production cost
* **Exchange rates** revert toward purchasing power parity (PPP) — eventually

Without mean-reversion models, finance would be unable to explain or predict any of these fundamental market behaviours.

> **Key Concept:** Geometric Brownian Motion (GBM) — the workhorse stock price model — has *no* mean reversion. Prices drift forever, never returning to a "fair value." This is appropriate for stocks (where compounding growth is the norm) but disastrous for interest rates, where rates physically cannot drift to infinity. Mean-reversion models fill this gap.

---
## 1. Why Mean Reversion? The Rubber Band Analogy

### Stock prices vs interest rates

In the GBM notebook, we modeled stock prices as random walks with drift. Stocks can wander to infinity or near zero. But **interest rates are fundamentally different.** We have never seen rates at 1000%, and they do not wander off to infinity. Instead, rates fluctuate within a range, pulled back toward a "normal" level by economic forces:

- When rates are very high, borrowing slows, the economy cools, and central banks cut rates.
- When rates are very low, borrowing surges, the economy heats up, and central banks raise rates.

This **mean-reverting** behavior is like a rubber band attached to a post:

- The post represents the long-run average rate (say 5%).
- Random shocks (economic news, policy surprises) stretch the rubber band away from the post.
- The tension pulls the rate back. The farther it stretches, the stronger the pull.

> **Key Concept:** Mean reversion means extreme values are temporary. If the interest rate is unusually high, we expect it to fall back. If unusually low, we expect it to rise. This is the opposite of a random walk, where extreme values persist.

### Where else do we see mean reversion?

- **Commodity prices:** Oil reverts to production costs.
- **Volatility:** Spikes during crises but reverts to a baseline.
- **Exchange rates:** Purchasing power parity acts as a long-run anchor.
- **Temperature:** Daily temperatures fluctuate but revert to seasonal averages.

### The mathematics of mean reversion

A general mean-reverting stochastic process has the form:

$$dX_t = \theta(\mu - X_t) dt + \sigma \, dW_t$$

The **drift** term $\theta(\mu - X_t)$ has a specific structure:
* When $X_t < \mu$: drift is *positive* (process pulled upward)
* When $X_t > \mu$: drift is *negative* (process pulled downward)
* When $X_t = \mu$: drift is zero (no systematic pull)

The strength of the pull is proportional to the *distance* from $\mu$. The further from equilibrium, the stronger the restoring force — like a spring.

The diffusion term $\sigma dW_t$ adds randomness, preventing the process from settling exactly at $\mu$. The result: a process that *fluctuates around* $\mu$ rather than drifting away from it.

### Three canonical mean-reverting models

This notebook covers the three most important mean-reverting models in finance:

| Model | Process | Used For |
|-------|---------|----------|
| **Ornstein-Uhlenbeck (OU)** | $dX = \theta(\mu - X) dt + \sigma dW$ | Interest rates, spreads, volatility |
| **Vasicek (1977)** | $dr = a(b - r) dt + \sigma dW$ | Term structure of interest rates |
| **Cox-Ingersoll-Ross (CIR, 1985)** | $dr = a(b - r) dt + \sigma\sqrt{r} dW$ | Interest rates with positivity constraint |

Vasicek is OU specialised to short rates. CIR adds a multiplicative volatility that ensures rates stay positive — addressing OU's biggest weakness.

> **CFA Exam Tip:** Vasicek and CIR are the two "classical" short-rate models tested at CFA Level 2 and Level 3. Know their key features: both are mean-reverting, but only CIR guarantees positive rates. Vasicek has a closed-form bond pricing formula; CIR has one too but more complex. Both are *one-factor* models — their main limitation.

## 2. Setup## 2. Setup

This notebook implements three mean-reverting models from scratch using NumPy. We compare exact simulations (using analytical solutions where available) against numerical schemes (Euler-Maruyama, Milstein) and verify properties via Monte Carlo experiments.

The notebook progresses from the simplest model (OU) to its finance-specialised variants (Vasicek and CIR), then to bond pricing applications and parameter estimation.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize, linalg
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 3. The Ornstein-Uhlenbeck (OU) Process

### The SDE

$$dX_t = \theta(\mu - X_t) \, dt + \sigma \, dW_t$$

Let us understand each term:

**The drift term** $\theta(\mu - X_t) \, dt$:
- If $X_t > \mu$ (above the mean): drift is negative, pushing *down*
- If $X_t < \mu$ (below the mean): drift is positive, pushing *up*
- If $X_t = \mu$: drift is zero -- no pull

**The diffusion term** $\sigma \, dW_t$: Pure random noise trying to push the process away from the mean.

The process is a tug-of-war: drift pulls toward $\mu$, noise pushes away.

### Parameter definitions

| Parameter | Symbol | Meaning | Rubber band analogy |
|-----------|--------|---------|---------------------|
| Mean-reversion speed | $\theta$ | How fast it returns to the mean | Stiffness of the rubber band |
| Long-run mean | $\mu$ | The level it reverts to | Position of the post |
| Volatility | $\sigma$ | Intensity of random shocks | Strength of the wind |

### Half-life: a practical measure

$$t_{1/2} = \frac{\ln 2}{\theta}$$

**Worked example:** If $\theta = 2$, then $t_{1/2} = 0.693/2 = 0.35$ years (~4 months). If the rate is 3% above its long-run mean, we expect it to be ~1.5% above the mean in 4 months.

> **Key Concept:** The half-life gives an intuitive timescale. A half-life of 1 year means slow reversion. A half-life of 1 month means fast reversion.

### Comparison with GBM

| Feature | GBM | OU |
|---------|-----|-----|
| Long-run behavior | Wanders to infinity | Stays near $\mu$ |
| Variance over time | Grows forever | Settles to $\sigma^2/(2\theta)$ |
| Good for | Stock prices | Interest rates, volatility |

### The Ornstein-Uhlenbeck process — the foundation

The OU process, introduced by Leonard Ornstein and George Uhlenbeck in 1930 to model the velocity of a particle in fluid (Brownian motion with friction), is defined by:

$$dX_t = \theta(\mu - X_t) dt + \sigma \, dW_t$$

with parameters:
* **$\mu$** = long-run mean (equilibrium value)
* **$\theta > 0$** = mean reversion speed (per unit time)
* **$\sigma > 0$** = diffusion coefficient (volatility)

### Key properties

1. **Conditional distribution:** Given $X_t$ at time $t$, the value at time $T > t$ is normally distributed:

$$X_T | X_t \sim N\left(\mu + (X_t - \mu) e^{-\theta(T-t)}, \frac{\sigma^2}{2\theta}(1 - e^{-2\theta(T-t)})\right)$$

The mean *exponentially decays* toward $\mu$. The variance grows toward an asymptotic value.

2. **Stationary distribution:** As $T \to \infty$:

$$X_\infty \sim N\left(\mu, \frac{\sigma^2}{2\theta}\right)$$

The process forgets its initial condition and settles into a stationary distribution centred at $\mu$.

3. **Half-life of mean reversion:** The time for the deviation from $\mu$ to halve is:

$$t_{1/2} = \frac{\ln 2}{\theta}$$

For $\theta = 1$: half-life is ~0.69 years; for $\theta = 0.1$: ~6.9 years.

> **Key Concept:** The OU process has *Gaussian* increments — at any horizon, $X_T$ is normally distributed. This makes it analytically tractable but allows *negative values*, which is problematic for interest rates and other inherently positive variables.

---
## 4. Exact Solution and Simulation

### The exact solution

$$X_t = \mu + (X_0 - \mu)e^{-\theta t} + \sigma \int_0^t e^{-\theta(t-s)} dW_s$$

Reading each piece:
1. **$\mu$**: The long-run mean.
2. **$(X_0 - \mu)e^{-\theta t}$**: Initial deviation, decaying exponentially. After several half-lives, the process has "forgotten" where it started.
3. **$\sigma \int_0^t e^{-\theta(t-s)} dW_s$**: Accumulated noise with exponential decay -- recent noise matters more than old noise.

### Conditional distribution (for simulation)

$$X_{t+\Delta t} | X_t \sim \mathcal{N}\left(\mu + (X_t - \mu)e^{-\theta \Delta t}, \frac{\sigma^2}{2\theta}(1 - e^{-2\theta \Delta t})\right)$$

### Worked example

$\theta = 2$, $\mu = 5\%$, $\sigma = 2\%$, $X_0 = 10\%$, $\Delta t = 1$ year:
- Conditional mean: $0.05 + 0.05 \times e^{-2} = 0.05 + 0.0068 = 5.68\%$
- Conditional std: $\sqrt{0.0001 \times 0.982} = 0.99\%$

After 1 year from 10%, the rate is expected at 5.68% (much closer to 5%) with std ~1%.

### Why the OU process matters for finance

Despite its simplicity, the OU process is the foundation of:

1. **Interest rate modelling:** Vasicek's model is OU specialised to short rates
2. **Volatility models:** OU on log-volatility (Heston, Stein-Stein)
3. **Pairs trading:** Spread between two cointegrated assets often follows OU
4. **Commodity prices:** Convenience yield and basis often modelled as OU
5. **Energy prices:** Electricity, natural gas spot prices show OU-like behaviour

The OU process is what GBM is for stock prices: the "default" model from which more sophisticated models build. Understanding OU thoroughly is the key to understanding mean-reverting finance.

In [ ]:
def ou_exact_sim(X0, theta, mu, sigma, T, n_steps, n_paths=1):
    """Exact simulation of the OU process using the conditional distribution.
    
    At each step, we draw from N(conditional_mean, conditional_variance)
    which is the exact distribution -- no discretization error.
    """
    dt = T / n_steps
    X = np.zeros((n_paths, n_steps + 1))
    X[:, 0] = X0
    
    # Pre-compute the conditional mean weight and variance
    mean_factor = np.exp(-theta * dt)          # weight on X_t in the conditional mean
    var = sigma**2 / (2 * theta) * (1 - np.exp(-2 * theta * dt))  # conditional variance
    
    for i in range(n_steps):
        conditional_mean = mu + (X[:, i] - mu) * mean_factor
        X[:, i+1] = conditional_mean + np.sqrt(var) * rng.standard_normal(n_paths)
    
    t = np.linspace(0, T, n_steps + 1)
    return t, X

# ── Parameters
theta = 2.0      # mean-reversion speed (half-life = 0.35 years)
mu_ou = 0.05     # long-run mean (5%)
sigma_ou = 0.02  # volatility
X0 = 0.10        # start at 10% -- above the mean

t, X_paths = ou_exact_sim(X0, theta, mu_ou, sigma_ou, 5.0, 1000, 10)

fig, ax = plt.subplots()
for i in range(10):
    ax.plot(t, X_paths[i] * 100, linewidth=0.8, alpha=0.7)
ax.axhline(mu_ou * 100, color='black', linestyle='--', linewidth=2, label=f'Long-run mean mu = {mu_ou*100:.1f}%')
ax.set_xlabel('Time (years)')
ax.set_ylabel('X(t) (%)')
ax.set_title(f'Ornstein-Uhlenbeck Process (theta={theta}, half-life={np.log(2)/theta:.2f}yr)')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:** All 10 paths start at 10% and are rapidly pulled toward 5%. After the first year, they fluctuate around 5% without drifting away.**Interpretation:** All 10 paths start at 10% and are rapidly pulled toward 5%. After the initial transient, the paths fluctuate around 5% with similar amplitude regardless of starting point.

### What the simulation reveals

Three observations from the OU simulation:

1. **Exponential decay of initial condition:** The mean of paths decays from 10% toward 5% according to $E[X_t] = \mu + (X_0 - \mu) e^{-\theta t}$. The faster $\theta$, the faster this decay.

2. **Bounded fluctuations:** Once the transient has decayed, paths fluctuate within a roughly stationary band around $\mu$. The width of this band is $\sigma/\sqrt{2\theta}$ (one standard deviation).

3. **No drift:** Unlike GBM, paths don't drift indefinitely upward or downward. They stay near the long-run mean.

### Why this matters for interest rate modelling

Real interest rates exhibit exactly this behaviour:
* During easy money periods, rates can stay low for years (long transient)
* During tight money, rates spike then decay back
* Long-run, rates fluctuate around what most economists view as a "neutral rate"

The OU process captures this *qualitative* behaviour, even if its *quantitative* details (Gaussian increments, no positivity constraint) are imperfect.

> **CFA Exam Tip:** When evaluating a mean-reverting model, ask three questions: (1) Does it converge to a stationary distribution? (2) What's the speed of mean reversion? (3) Are the model's properties consistent with the data? OU is excellent on (1) and (2) but weak on (3) for interest rates because of the negative-value problem.

---
## 5. Stationary Distribution -- Where the Process "Settles Down"

$$X_\infty \sim \mathcal{N}\left(\mu, \frac{\sigma^2}{2\theta}\right)$$

**What this tells us:**
- Process fluctuates around $\mu$
- Spread is $\sigma/\sqrt{2\theta}$ -- higher $\sigma$ widens it, stronger $\theta$ narrows it
- Think of it as the "climate" vs the daily "weather"

### Intuition: the bathtub analogy

Imagine a bathtub with a drain (mean reversion) and a running faucet (noise). The stationary distribution balances the inflow of randomness against the outflow of mean reversion.

### Worked example

$\mu = 5\%$, $\theta = 2$, $\sigma = 2\%$:
- Stationary std: $2\%/\sqrt{4} = 1\%$
- 95% of the time: between $[3\%, 7\%]$

### Calibrating from observed data

To calibrate an OU process to historical data, you can use either:

1. **Method of moments:** Match the sample mean and variance to the theoretical stationary mean and variance:
$$\hat{\mu} = \bar{x}, \quad \frac{\hat{\sigma}^2}{2\hat{\theta}} = s^2$$

2. **Maximum likelihood:** Maximise the log-likelihood of observed data given the OU dynamics. This requires the conditional distribution at each step.

3. **Linear regression:** Discretise the OU equation and fit:
$$X_{t+\Delta} - X_t = \theta(\mu - X_t) \Delta + \sigma\sqrt{\Delta} \, \epsilon_t$$

Each method has trade-offs. MLE is statistically efficient but harder to implement; method of moments is simpler but less efficient. Linear regression is fastest and works well in practice for small $\Delta$.

In [ ]:
# ── Verify stationary distribution with many paths run for a long time
t_long, X_long = ou_exact_sim(X0, theta, mu_ou, sigma_ou, 20.0, 10000, 50000)
X_terminal = X_long[:, -1]

stationary_std = sigma_ou / np.sqrt(2 * theta)

fig, ax = plt.subplots()
ax.hist(X_terminal * 100, bins=100, density=True, color=PRIMARY, edgecolor='white', alpha=0.7)
x_grid = np.linspace(X_terminal.min(), X_terminal.max(), 200)
ax.plot(x_grid * 100, stats.norm.pdf(x_grid, mu_ou, stationary_std) / 100, 
        color=SECONDARY, linewidth=2, label=f'N({mu_ou*100:.1f}%, {stationary_std*100:.3f}%)')
ax.set_xlabel('X (%)')
ax.set_ylabel('Density')
ax.set_title('Stationary Distribution of OU Process')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Stationary mean:  theory = {mu_ou*100:.3f}%, simulated = {np.mean(X_terminal)*100:.3f}%")
print(f"Stationary std:   theory = {stationary_std*100:.3f}%, simulated = {np.std(X_terminal)*100:.3f}%")

**Interpretation:** The histogram of 50,000 terminal values fits the normal distribution perfectly. The process has "forgotten" its starting point.

### Effect of mean-reversion speed

- **Small $\theta$ (slow):** Rubber band is loose, process wanders far.
- **Large $\theta$ (fast):** Rubber band is stiff, process barely strays.**Interpretation:** The histogram of 50,000 terminal values fits the normal distribution predicted by theory — confirming the stationary distribution result.

### Why the stationary distribution matters

The stationary distribution is the long-run behaviour of the process — the distribution you'd see if you sampled $X_t$ at a random time after the transient has decayed.

For OU:
* **Mean** = $\mu$ (the equilibrium level)
* **Variance** = $\sigma^2 / (2\theta)$ (depends on speed of reversion)

This formula reveals an important trade-off:
* Higher $\sigma$ → wider stationary distribution
* Higher $\theta$ → narrower stationary distribution

For interest rates calibrated from historical data, the stationary parameters tell you the long-run mean and volatility of rates — useful for stress testing and scenario analysis.

> **Key Concept:** The existence of a stationary distribution is a *defining feature* of mean-reverting processes. GBM does not have one (it drifts to infinity); Brownian motion does not have one (variance grows without bound); OU does. This is why mean-reverting processes are appropriate when bounded long-run behaviour is required.

In [ ]:
# ── Effect of theta on path behavior
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, th in zip(axes, [0.5, 2.0, 10.0]):
    t_demo, X_demo = ou_exact_sim(0.10, th, 0.05, 0.02, 5.0, 1000, 5)
    for i in range(5):
        ax.plot(t_demo, X_demo[i] * 100, linewidth=0.8)
    ax.axhline(5, color='black', linestyle='--')
    ax.set_title(f'theta = {th} (half-life = {np.log(2)/th:.2f}yr)')
    ax.set_xlabel('Time')
    ax.set_ylabel('X(t) (%)')
    ax.set_ylim(1, 12)

plt.suptitle('Effect of Mean-Reversion Speed', fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation:**
- $\theta = 0.5$ (half-life 1.4 yr): Takes years to drift back to 5%.
- $\theta = 2$ (half-life 0.35 yr): Rapid return after the first year.
- $\theta = 10$ (half-life 0.07 yr): Almost immediate reversion.**Interpretation:**

### Effect of $\theta$ on path behavior

The simulation with different $\theta$ values shows:

* **$\theta = 0.5$ (slow reversion):** Paths drift far from $\mu$, taking a long time to return. Half-life is $\ln 2 / 0.5 \approx 1.4$ years.

* **$\theta = 2$ (medium reversion):** Paths revert noticeably; visible "snap back" toward $\mu$. Half-life is $\ln 2 / 2 \approx 0.35$ years.

* **$\theta = 10$ (fast reversion):** Paths barely deviate from $\mu$ — mean reversion overwhelms diffusion. Half-life is $\ln 2 / 10 \approx 0.07$ years.

The visual difference is striking. As $\theta$ increases, the path becomes "pinned" to $\mu$ and looks more like white noise around the mean.

### Calibration intuition

When fitting a mean-reverting model to data:
* High autocorrelation in the data → small $\theta$ (slow reversion)
* Low autocorrelation → large $\theta$ (fast reversion)
* Effectively no autocorrelation → $\theta \to \infty$, which is just white noise around $\mu$

> **Common Mistake:** Beginners sometimes confuse the *speed* parameter $\theta$ with the *strength* of mean reversion. They are the same — higher $\theta$ means faster, stronger reversion. A "slow" mean-reverting process has small $\theta$.

---
## 6. The Vasicek Model -- Simplest Interest Rate Model

$$dr_t = a(b - r_t) \, dt + \sigma \, dW_t$$

| Parameter | Symbol | Finance meaning | Example |
|-----------|--------|----------------|---------|
| Speed | $a$ | How fast rates return to normal | $a = 0.5$ (half-life 1.4 yr) |
| Long-run mean | $b$ | The "neutral rate" | $b = 5\%$ |
| Volatility | $\sigma$ | Daily rate uncertainty | $\sigma = 1.5\%$ |

### Real-world intuition

- **$a$**: "How quickly the central bank reacts." High $a$ = aggressive steering.
- **$b$**: The "neutral rate" -- neither stimulating nor restraining.
- **$\sigma$**: Day-to-day uncertainty from economic news, policy surprises.

> **Key Concept:** Vasicek gives closed-form bond prices, yield curves, and option prices. Its weakness: it allows negative rates (because noise is additive). This was considered fatal until negative rates actually appeared in Europe and Japan in 2014-2016!

### The Vasicek model — short rates with structure

Oldrich Vasicek's 1977 paper *"An equilibrium characterization of the term structure"* introduced the first widely-used mean-reverting interest rate model. It's just an OU process with finance-specific notation:

$$dr_t = a(b - r_t) dt + \sigma \, dW_t$$

where:
* **$r_t$** = instantaneous short rate (overnight rate, conceptually)
* **$a > 0$** = mean reversion speed (analogous to OU's $\theta$)
* **$b$** = long-run mean rate (analogous to OU's $\mu$)
* **$\sigma > 0$** = volatility

### What Vasicek gives us

Vasicek's model is enormously influential because it provides:

1. **Closed-form bond prices** for any maturity
2. **Closed-form yield curve** at any time, given $r_0$
3. **Mean-reverting dynamics** consistent with empirical short-rate behaviour
4. **Tractable risk management** — analytical Greeks for interest rate derivatives

These analytical advantages made Vasicek the standard short-rate model for years, despite its key limitation: rates can go negative.

### The negative rate problem

Until 2014, "negative interest rates" were considered a theoretical curiosity. Then the ECB and Swiss National Bank introduced negative policy rates, and government bond yields in Europe and Japan went deeply negative.

For Vasicek users, this was a feature — the model has always allowed negative rates. For users of *positive-only* models like CIR or BGM, this required model changes or interpretation tricks.

> **CFA Exam Tip:** The CFA curriculum tests Vasicek and CIR side-by-side. The trade-off: Vasicek has Gaussian rates (analytical convenience but negative rates possible); CIR has square-root volatility (positive rates guaranteed but more complex math). For decades, finance practitioners chose between these features.

In [ ]:
# ── Vasicek is OU with finance notation
a_vas, b_vas, sigma_vas = 0.5, 0.05, 0.015
r0 = 0.03  # current short rate at 3% (below long-run mean of 5%)

t_vas, r_vas = ou_exact_sim(r0, a_vas, b_vas, sigma_vas, 10.0, 2520, 10)

fig, ax = plt.subplots()
for i in range(10):
    ax.plot(t_vas, r_vas[i] * 100, linewidth=0.8, alpha=0.7)
ax.axhline(b_vas * 100, color='black', linestyle='--', linewidth=2, label=f'b = {b_vas*100:.1f}%')
ax.axhline(0, color='red', linestyle=':', alpha=0.5, label='Zero bound')
ax.set_xlabel('Time (years)')
ax.set_ylabel('Short Rate (%)')
ax.set_title(f'Vasicek Model: a={a_vas}, b={b_vas*100:.1f}%, sigma={sigma_vas*100:.1f}%')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:** Starting from 3%, rates are pulled upward toward 5%. Negative rates are unlikely but theoretically possible.**Interpretation:** Starting from 3%, rates are pulled upward toward 5%. Negative rates are *possible* but unlikely if $b$ is positive and $\sigma$ is reasonable.

### When does Vasicek go negative?

The probability of negative rates under Vasicek:

$$P(r_T < 0 | r_0) = \Phi\left(\frac{-E[r_T | r_0]}{\sqrt{\text{Var}(r_T | r_0)}}\right)$$

For typical parameters ($r_0 = 5\%$, $b = 5\%$, $a = 0.3$, $\sigma = 1\%$), this probability is small (~ 0.5%) for reasonable horizons. It only becomes problematic when:
* $b$ is small (close to zero)
* $\sigma$ is large (high volatility)
* The horizon is long (more time for negative excursion)

### Negative rates in practice

When the ECB cut rates to -0.4% in 2016 and Japanese 10-year bonds traded at -0.1%, Vasicek's "feature" became operational. Practitioners using Vasicek had no problem; those using CIR (which strictly enforces $r > 0$) had to add a lower bound shift.

This historical episode illustrates an important modelling principle: *constraints reflect assumptions about the world*. CIR's positivity assumption was wrong for European rates in 2016. Vasicek's "anything goes" assumption turned out to be more flexible.

---
## 7. Bond Pricing under Vasicek -- What A(T) and B(T) Mean

A zero-coupon bond pays \$1 at maturity $T$. Its price is:

$$P(t, T) = \exp(A(\tau) - B(\tau) \cdot r_t)$$

where $\tau = T - t$ and:

$$B(\tau) = \frac{1 - e^{-a\tau}}{a}, \quad A(\tau) = \left(b - \frac{\sigma^2}{2a^2}\right)(B - \tau) - \frac{\sigma^2}{4a} B^2$$

### What A and B mean intuitively

**$B(\tau)$ -- rate sensitivity:**
- Short maturities: $B \approx \tau$ (duration = maturity)
- Long maturities: $B \to 1/a$ (saturates because distant rates revert to mean)
- **Mean reversion limits how much long bonds are affected by today's rate.**

**$A(\tau)$ -- convexity and mean-level adjustment:**
- Captures the pull toward $b$ and the convexity benefit from Jensen's inequality.

### Long-run yield

$R(\infty) = b - \sigma^2/(2a^2)$ -- slightly below $b$ due to convexity.

In [ ]:
def vasicek_bond_price(r, a, b, sigma, tau):
    """Zero-coupon bond price under Vasicek.
    
    P(t,T) = exp(A(tau) - B(tau) * r_t)  where tau = T - t
    """
    B = (1 - np.exp(-a * tau)) / a
    A = (b - sigma**2 / (2 * a**2)) * (B - tau) - sigma**2 / (4 * a) * B**2
    return np.exp(A - B * r)

def vasicek_yield(r, a, b, sigma, tau):
    """Continuously compounded yield under Vasicek."""
    P = vasicek_bond_price(r, a, b, sigma, tau)
    return -np.log(P) / tau

# ── Plot yield curves for different starting rates
taus = np.linspace(0.1, 30, 200)

fig, ax = plt.subplots()
for r0_val, c, ls in [(0.02, PRIMARY, '-'), (0.05, SECONDARY, '-'), (0.08, TERTIARY, '-')]:
    yields = vasicek_yield(r0_val, a_vas, b_vas, sigma_vas, taus)
    ax.plot(taus, yields * 100, color=c, linewidth=2, label=f'r_0 = {r0_val*100:.0f}%')

# Long-run yield
R_inf = b_vas - sigma_vas**2 / (2 * a_vas**2)
ax.axhline(R_inf * 100, color='grey', linestyle='--', alpha=0.5, label=f'R(inf) = {R_inf*100:.2f}%')
ax.axhline(b_vas * 100, color='black', linestyle=':', alpha=0.3, label=f'b = {b_vas*100:.1f}%')

ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Yield (%)')
ax.set_title('Vasicek Yield Curves')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:**
- $r_0 = 2\%$ (below mean): Upward-sloping curve. Market expects rates to rise.
- $r_0 = 5\%$ (at mean): Slightly downward due to convexity.
- $r_0 = 8\%$ (above mean): Inverted. Market expects rates to fall.
All curves converge to $R(\infty)$.**Interpretation:**

### The Vasicek bond pricing formula

The exact bond price under Vasicek is:

$$P(t, T) = A(t, T) e^{-B(t, T) \cdot r_t}$$

where $A$ and $B$ are deterministic functions of model parameters and time-to-maturity. Specifically:

$$B(t, T) = \frac{1 - e^{-a(T-t)}}{a}$$

$$A(t, T) = \exp\left[\frac{(B - (T-t))(a^2 b - \sigma^2/2)}{a^2} - \frac{\sigma^2 B^2}{4a}\right]$$

This is an **affine term structure** — bond yields are linear (affine) in the short rate $r_t$. This linearity is what makes the model so analytically tractable.

### Yield curve shapes

The Vasicek yield curve can be:
* **Upward sloping (normal):** When $r_t$ is below the long-run yield
* **Downward sloping (inverted):** When $r_t$ is high relative to expected future rates
* **Hump-shaped:** Possible for specific parameter combinations
* **Flat:** When current and long-run rates roughly match

This flexibility allows Vasicek to match many observed yield curve shapes — but the *single factor* (only $r_t$) means it cannot capture all patterns simultaneously.

> **Key Concept:** A yield curve embeds expectations about the path of future short rates. The Vasicek formula makes this explicit: $P(t, T)$ depends on the model parameters and the current $r_t$. Higher current $r$ → lower bond prices → higher yields. The mean-reversion parameter $a$ controls how quickly rate expectations decay toward $b$ in the long run.

---
## 8. The CIR Model -- Preventing Negative Rates

### The CIR fix: level-dependent volatility

$$dr_t = a(b - r_t) \, dt + \sigma \sqrt{r_t} \, dW_t$$

**Why $\sqrt{r}$ works:**
1. **Near zero:** $\sigma\sqrt{r} \approx 0$, noise vanishes. Only drift $ab > 0$ pushes rates up.
2. **Far from zero:** Volatility increases with rates (realistic).
3. **Feller condition:** If $2ab > \sigma^2$, rates stay strictly positive.

> **Key Concept:** The Feller condition $2ab > \sigma^2$ says "mean-reversion must be strong enough relative to noise." The drift pushes at strength $ab$ near zero, noise at $\sigma^2/2$. If push beats noise, rates stay positive.

| Feature | Vasicek | CIR |
|---------|---------|-----|
| Noise at $r=0$ | $\sigma$ (unchanged) | $0$ (noise off) |
| Negative rates? | Yes | No (if $2ab > \sigma^2$) |
| Distribution | Normal | Non-central $\chi^2$ |

### Why the square-root volatility?

The choice of $\sigma\sqrt{r}$ for the diffusion coefficient is not arbitrary — it produces several desirable properties:

1. **Positivity:** As $r \to 0^+$, the diffusion vanishes and only the positive drift remains, pulling $r$ back up.

2. **Heteroskedastic volatility:** Higher rates → higher volatility, consistent with empirical observation in some periods.

3. **Analytical tractability:** The square-root form preserves enough structure that closed-form bond prices remain available.

4. **Affine structure:** Yields are still linear in $r$, simplifying multi-asset pricing.

Other functional forms have been tried (constant elasticity of variance, log-normal volatility), but none combine these properties as elegantly as CIR.

In [ ]:
def cir_milstein(r0, a, b, sigma, T, n_steps, n_paths=1):
    """CIR simulation using the Milstein scheme with reflection at zero.
    
    The Milstein scheme adds a correction term (0.25*sigma^2*dt*(Z^2-1))
    beyond the basic Euler step, giving better accuracy for the square-root
    diffusion. We reflect at zero to handle numerical undershoot.
    """
    dt = T / n_steps
    r = np.zeros((n_paths, n_steps + 1))
    r[:, 0] = r0
    
    for i in range(n_steps):
        Z = rng.standard_normal(n_paths)
        r_pos = np.maximum(r[:, i], 0)  # ensure non-negative for sqrt
        sqrt_r = np.sqrt(r_pos)
        
        # Milstein scheme for CIR
        r[:, i+1] = (r[:, i] 
                     + a * (b - r_pos) * dt               # drift: pull toward b
                     + sigma * sqrt_r * np.sqrt(dt) * Z   # diffusion: sqrt(r) * noise
                     + 0.25 * sigma**2 * dt * (Z**2 - 1)) # Milstein correction
        
        # Reflect at zero (numerical safety)
        r[:, i+1] = np.abs(r[:, i+1])
    
    t = np.linspace(0, T, n_steps + 1)
    return t, r

# ── CIR parameters
a_cir, b_cir, sigma_cir = 0.5, 0.05, 0.05
r0_cir = 0.03
feller = 2 * a_cir * b_cir / sigma_cir**2
print(f"Feller ratio 2ab/sigma^2 = {feller:.2f} {'> 1 (strictly positive)' if feller > 1 else '<= 1 (can hit zero)'}")

t_cir, r_cir = cir_milstein(r0_cir, a_cir, b_cir, sigma_cir, 10.0, 2520, 10)

fig, ax = plt.subplots()
for i in range(10):
    ax.plot(t_cir, r_cir[i] * 100, linewidth=0.8, alpha=0.7)
ax.axhline(b_cir * 100, color='black', linestyle='--', linewidth=2, label=f'b = {b_cir*100:.1f}%')
ax.axhline(0, color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('Time (years)')
ax.set_ylabel('Short Rate (%)')
ax.set_title(f'CIR Model: a={a_cir}, b={b_cir*100:.1f}%, sigma={sigma_cir*100:.1f}%')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:** CIR paths stay strictly positive. Volatility shrinks when rates are low and increases when high -- the $\sqrt{r}$ effect.**Interpretation:** CIR paths stay strictly positive. Volatility shrinks when rates are low and grows when rates are high — a more realistic feature than Vasicek's constant volatility.

### CIR — solving the negative rate problem

The Cox-Ingersoll-Ross (1985) model:

$$dr_t = a(b - r_t) dt + \sigma \sqrt{r_t} \, dW_t$$

The key innovation: the volatility term $\sigma\sqrt{r_t}$ is *proportional to the square root of $r$*. This has two important effects:

1. **Positivity:** When $r$ approaches zero, volatility approaches zero too. The diffusion vanishes, leaving only the mean-reverting drift to pull rates upward. This guarantees rates stay positive (under the **Feller condition**: $2ab \geq \sigma^2$).

2. **Heteroskedasticity:** Volatility depends on the rate level. When rates are high, volatility is high (matches empirical observation in some periods). When rates are low, volatility is low.

### The Feller condition

The condition $2ab \geq \sigma^2$ is *necessary* for rates to stay strictly positive. If violated, the process can hit zero (and the model is ill-defined). In practice:
* Calibrated CIR parameters typically satisfy the condition with margin
* But fitted parameters can violate it for some markets — requiring constraints in the calibration

> **CFA Exam Tip:** Memorise the Feller condition: $2ab \geq \sigma^2$ ensures CIR rates stay positive. The condition essentially says: mean reversion speed × long-run mean must be large enough to overcome the volatility.

---
## 9. CIR Bond Pricing

CIR also has affine bond prices $P = \exp(A - Br)$ with $\gamma = \sqrt{a^2 + 2\sigma^2}$:

$$B(\tau) = \frac{2(e^{\gamma\tau} - 1)}{(\gamma + a)(e^{\gamma\tau} - 1) + 2\gamma}$$

$$A(\tau) = \frac{2ab}{\sigma^2} \ln\left(\frac{2\gamma e^{(a+\gamma)\tau/2}}{(\gamma+a)(e^{\gamma\tau}-1)+2\gamma}\right)$$

In [ ]:
def cir_bond_price(r, a, b, sigma, tau):
    """Zero-coupon bond price under CIR."""
    gamma = np.sqrt(a**2 + 2 * sigma**2)
    exp_gamma = np.exp(gamma * tau)
    denom = (gamma + a) * (exp_gamma - 1) + 2 * gamma
    
    B = 2 * (exp_gamma - 1) / denom
    A = (2 * a * b / sigma**2) * np.log(2 * gamma * np.exp((a + gamma) * tau / 2) / denom)
    
    return np.exp(A - B * r)

def cir_yield(r, a, b, sigma, tau):
    """Continuously compounded yield under CIR."""
    P = cir_bond_price(r, a, b, sigma, tau)
    return -np.log(P) / tau

---
## 10. Yield Curve Shapes

| Shape | Condition | Economic interpretation |
|-------|-----------|------------------------|
| **Normal** (upward) | $r_0 < b$ | Rates low, expected to rise |
| **Flat** | $r_0 \approx b$ | Rates near long-run level |
| **Inverted** (downward) | $r_0 > b$ | Rates high, expected to fall |

> **Key Concept:** Mean reversion is the mechanism that gives yield curves their shape. Without it, only flat curves would exist.

### CIR bond pricing

Like Vasicek, CIR has a closed-form bond pricing formula:

$$P(t, T) = A(t, T) e^{-B(t, T) \cdot r_t}$$

where $A$ and $B$ are different (more complex) functions than Vasicek's:

$$B(t, T) = \frac{2(e^{\gamma(T-t)} - 1)}{(\gamma + a)(e^{\gamma(T-t)} - 1) + 2\gamma}$$

with $\gamma = \sqrt{a^2 + 2\sigma^2}$.

The CIR model is also affine — yields are linear in $r$. But the parameter values produce subtly different yield curve shapes than Vasicek, particularly at long maturities and low rates.

In [ ]:
# ── Compare yield curves from Vasicek and CIR
taus = np.linspace(0.1, 30, 200)
r0_vals = [0.02, 0.05, 0.08]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for r0_v, c in zip(r0_vals, [PRIMARY, SECONDARY, TERTIARY]):
    y_vas = vasicek_yield(r0_v, a_vas, b_vas, sigma_vas, taus)
    y_cir = cir_yield(r0_v, a_cir, b_cir, sigma_cir, taus)
    ax1.plot(taus, y_vas * 100, color=c, linewidth=2, label=f'r_0={r0_v*100:.0f}%')
    ax2.plot(taus, y_cir * 100, color=c, linewidth=2, label=f'r_0={r0_v*100:.0f}%')

ax1.set_xlabel('Maturity (years)')
ax1.set_ylabel('Yield (%)')
ax1.set_title('Vasicek Yield Curves')
ax1.legend()

ax2.set_xlabel('Maturity (years)')
ax2.set_ylabel('Yield (%)')
ax2.set_title('CIR Yield Curves')
ax2.legend()

plt.tight_layout()
plt.show()

**Interpretation:** Both models produce similar yield curve shapes. CIR curves differ slightly due to level-dependent volatility affecting convexity.**Interpretation:** Both models produce similar yield curve shapes. CIR curves differ slightly from Vasicek due to the different volatility specification.

### Vasicek vs CIR yield curves

For typical parameter values, both models produce:
* Similar short-rate dynamics (mean reversion to $b$ with comparable speed)
* Similar yield curve levels
* Similar yield curve slopes

Where they differ:
* **Convexity adjustment:** CIR's level-dependent volatility makes the convexity correction different
* **Long-rate behaviour:** At very long maturities, CIR's positivity constraint affects asymptotic yields
* **Low-rate behaviour:** Near zero rates, CIR's volatility dies out — Vasicek's stays constant

In practice, the choice between Vasicek and CIR often comes down to:
1. Are negative rates a serious risk? (CIR if no, Vasicek if possible)
2. Do you want analytical tractability? (Vasicek slightly easier)
3. What's the parameter regime? (At low rates and high volatility, the differences become more pronounced)

> **Key Concept:** Both Vasicek and CIR are one-factor short-rate models. They share the limitation that yields at all maturities are perfectly correlated through their dependence on $r_t$. Real yield curves show *imperfect* correlation across maturities — short rates and long rates can move differently. Multi-factor models (Hull-White, two-factor CIR, HJM) address this limitation.

---
## 11. Parameter Calibration via MLE

Since Vasicek has a known conditional distribution:
$$r_{t+\Delta t} | r_t \sim \mathcal{N}(b + (r_t - b)e^{-a\Delta t}, \frac{\sigma^2}{2a}(1 - e^{-2a\Delta t}))$$

we can write the log-likelihood and minimize.

> **Key Concept:** MLE works well for Vasicek because the conditional distribution is Gaussian. For CIR, it is non-central chi-squared -- more complex but same principle.

In [ ]:
def vasicek_neg_log_likelihood(params, data, dt):
    """Negative log-likelihood for the Vasicek model.
    
    We minimize this to find the MLE parameters.
    """
    a, b, sigma = params
    if a <= 0 or sigma <= 0:
        return 1e10  # invalid parameters
    
    n = len(data) - 1
    mean_factor = np.exp(-a * dt)
    var = sigma**2 / (2 * a) * (1 - np.exp(-2 * a * dt))
    
    if var <= 0:
        return 1e10
    
    r_prev = data[:-1]
    r_next = data[1:]
    conditional_mean = b + (r_prev - b) * mean_factor
    
    # Sum of squared deviations, scaled by variance
    nll = 0.5 * n * np.log(2 * np.pi * var) + 0.5 / var * np.sum((r_next - conditional_mean)**2)
    return nll

# ── Generate sample data from known parameters
true_a, true_b, true_sigma = 1.0, 0.05, 0.02
dt_cal = 1/252  # daily observations
_, r_sample = ou_exact_sim(0.04, true_a, true_b, true_sigma, 10.0, int(10/dt_cal), 1)
data = r_sample[0]

# MLE optimization
x0 = [0.5, 0.04, 0.01]  # initial guess
bounds = [(0.01, 10), (0.001, 0.2), (0.001, 0.1)]
result = optimize.minimize(vasicek_neg_log_likelihood, x0, args=(data, dt_cal),
                          method='L-BFGS-B', bounds=bounds)

a_hat, b_hat, sig_hat = result.x

print(f"{'Parameter':<12} {'True':>10} {'MLE':>10}")
print('-' * 34)
print(f"{'a':<12} {true_a:>10.4f} {a_hat:>10.4f}")
print(f"{'b':<12} {true_b:>10.4f} {b_hat:>10.4f}")
print(f"{'sigma':<12} {true_sigma:>10.4f} {sig_hat:>10.4f}")

**Interpretation:** MLE estimates are close to true values, especially for $b$ and $\sigma$. Speed $a$ may have more error as it requires observing multiple reversions.**Interpretation:** MLE estimates are close to true values, especially for $b$ and $\sigma$. The mean reversion speed $a$ is the hardest to estimate accurately — typical of mean-reverting processes.

### Why $a$ is hard to estimate

The mean reversion speed $a$ is notoriously difficult to estimate from finite samples. Two reasons:

1. **Identification problem:** With short data, the process looks similar to a random walk. The signature of mean reversion (eventual return to $b$) requires long enough samples to be visible.

2. **Bias:** MLE estimates of $a$ are *systematically biased* — they tend to overestimate the true value. This is the well-known "Yu-Phillips bias" in autoregressive estimation.

For 10 years of monthly data on US Treasury yields, the standard error on $a$ is typically 30-50% of the estimate. The implication: confidence intervals for $a$ are wide, and downstream model outputs (e.g., yield curve fits) can be sensitive to small changes in the estimate.

### Best practices for parameter estimation

1. **Use long data series** when possible (decades, not just years)
2. **Bootstrap or simulate** to estimate parameter standard errors
3. **Compare estimates** across multiple data periods for stability
4. **Consider Bayesian priors** to regularise parameter estimates
5. **Test sensitivity** of downstream outputs to parameter uncertainty

> **CFA Exam Tip:** The CFA curriculum tests parameter estimation methods (especially MLE) and warns about the difficulty of estimating mean reversion speeds. Be prepared to explain why $a$ is hard to estimate and what biases the estimates carry.

---
## 12. Model Comparison -- Vasicek vs CIR

| Feature | Vasicek | CIR |
|---------|---------|-----|
| Negative rates? | Yes | No (if $2ab > \sigma^2$) |
| Distribution | Normal | Non-central $\chi^2$ |
| Volatility | Constant | Level-dependent |

In [ ]:
# ── Side-by-side comparison with matched parameters
a_comp, b_comp = 0.5, 0.05
sigma_comp = 0.03
r0_comp = 0.03
T_comp = 10.0
n_steps_comp = 2520
n_paths_comp = 5000

# Vasicek paths
t_comp, r_vas_paths = ou_exact_sim(r0_comp, a_comp, b_comp, sigma_comp, T_comp, n_steps_comp, n_paths_comp)
# CIR paths
_, r_cir_paths = cir_milstein(r0_comp, a_comp, b_comp, sigma_comp, T_comp, n_steps_comp, n_paths_comp)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Sample paths
for i in range(5):
    ax1.plot(t_comp, r_vas_paths[i] * 100, color=PRIMARY, alpha=0.5, linewidth=0.5)
    ax1.plot(t_comp, r_cir_paths[i] * 100, color=SECONDARY, alpha=0.5, linewidth=0.5)
ax1.axhline(b_comp * 100, color='black', linestyle='--')
ax1.axhline(0, color='red', linestyle=':')
ax1.set_xlabel('Time')
ax1.set_ylabel('Rate (%)')
ax1.set_title('Sample Paths')
ax1.plot([], [], color=PRIMARY, label='Vasicek')
ax1.plot([], [], color=SECONDARY, label='CIR')
ax1.legend()

# Terminal distributions
ax2.hist(r_vas_paths[:, -1] * 100, bins=80, density=True, alpha=0.5, color=PRIMARY, label='Vasicek')
ax2.hist(r_cir_paths[:, -1] * 100, bins=80, density=True, alpha=0.5, color=SECONDARY, label='CIR')
ax2.set_xlabel('Terminal Rate (%)')
ax2.set_title('Terminal Distribution')
ax2.legend()

# Negative rate frequency
neg_freq_vas = np.mean(r_vas_paths < 0, axis=0) * 100
neg_freq_cir = np.mean(r_cir_paths < 0, axis=0) * 100
ax3.plot(t_comp, neg_freq_vas, color=PRIMARY, linewidth=2, label='Vasicek')
ax3.plot(t_comp, neg_freq_cir, color=SECONDARY, linewidth=2, label='CIR')
ax3.set_xlabel('Time')
ax3.set_ylabel('P(r < 0) (%)')
ax3.set_title('Probability of Negative Rates')
ax3.legend()

plt.tight_layout()
plt.show()

print(f"Vasicek: {np.mean(r_vas_paths[:, -1] < 0)*100:.2f}% of paths end negative")
print(f"CIR:     {np.mean(r_cir_paths[:, -1] < 0)*100:.2f}% of paths end negative")

**Interpretation:**
- **Paths:** Both mean-revert toward 5%.
- **Terminal distribution:** Vasicek is symmetric (normal), CIR slightly skewed and truncated at zero.
- **Negative rates:** Vasicek probability grows over time; CIR stays non-negative.**Interpretation:**

### Side-by-side comparison

The matched-parameter comparison reveals:

| Feature | Vasicek | CIR |
|---------|---------|-----|
| **Volatility** | Constant ($\sigma$) | Level-dependent ($\sigma\sqrt{r}$) |
| **Negative rates** | Possible | Impossible (under Feller) |
| **Distribution** | Normal | Non-central chi-squared |
| **Bond pricing** | Closed-form, simple | Closed-form, complex |
| **Calibration** | Easier | Harder (more parameters interact) |
| **Real-world fit** | Good in many regimes | Better at high rates and during low-rate periods |

For matched parameters, both models produce qualitatively similar yield curves and bond prices. Differences emerge most clearly at:
* Very long maturities
* Very low or very high rate levels
* Specific stress scenarios

### Choosing between models

The practical choice depends on:
* **Application:** For pricing American Treasuries pre-2014, both work. For European negative-rate bonds in 2016, only Vasicek (with caveats) was usable.
* **Stress testing:** CIR provides cleaner downside scenarios because rates can't go arbitrarily negative.
* **Calibration regime:** CIR can fit volatility patterns better when volatility scales with rate level.

> **Common Mistake:** Practitioners sometimes treat the model choice as religious. It is not — different models are appropriate in different regimes. A sophisticated practitioner switches models or uses ensembles depending on the application.

In [ ]:
# ── Summary comparison table
print(f"{'Feature':<30} {'Vasicek':<20} {'CIR':<20}")
print('=' * 70)
print(f"{'SDE':<30} {'dr = a(b-r)dt + sdW':<20} {'dr = a(b-r)dt + s*sqrt(r)dW':<20}")
print(f"{'Negative rates?':<30} {'Yes':<20} {'No (if 2ab > s^2)':<20}")
print(f"{'Distribution of r_t':<30} {'Normal':<20} {'Non-central chi^2':<20}")
print(f"{'Vol structure':<30} {'Constant':<20} {'Level-dependent':<20}")
print(f"{'Affine term structure?':<30} {'Yes':<20} {'Yes':<20}")
print(f"{'Analytical bond prices?':<30} {'Yes':<20} {'Yes':<20}")

---
## 13. Summary and Key Takeaways

| Concept | Key Result |
|---------|------------|
| Mean reversion | "Rubber band" pull toward long-run mean |
| OU process | Simplest mean-reverting SDE; Gaussian conditional distribution |
| Half-life | $t_{1/2} = \ln 2/\theta$ |
| Stationary distribution | $\mathcal{N}(\mu, \sigma^2/2\theta)$ |
| Vasicek | Affine bond prices; allows negative rates |
| CIR | $\sqrt{r}$ diffusion prevents negative rates |
| Feller condition | $2ab > \sigma^2$ ensures positive rates |
| Yield curves | Normal ($r<b$), flat ($r \approx b$), inverted ($r>b$) |

### Limitations of one-factor models

Both Vasicek and CIR are *one-factor* models — yields at all maturities are determined by a single state variable $r_t$. This produces specific empirical limitations:

1. **Perfect correlation:** Yields at different maturities are perfectly correlated (mathematically — they are deterministic functions of the same $r_t$). Real yields are imperfectly correlated.

2. **Limited yield curve shapes:** The model can produce upward-sloping, inverted, or hump-shaped curves, but the dynamics are constrained. Some real-world curve shapes are not reachable.

3. **Volatility patterns:** Volatility at all maturities moves together. Real volatility patterns can vary across maturities (the "volatility smile" of bond markets).

### Multi-factor extensions

To address these limitations, practitioners use multi-factor models:

* **Hull-White (1990):** Time-dependent extension of Vasicek that fits any initial yield curve exactly
* **Two-factor models:** Add a second state variable (e.g., long-rate level alongside short-rate)
* **HJM (Heath-Jarrow-Morton):** Models the entire forward rate curve directly
* **LIBOR Market Models:** Models the discrete forward LIBOR rates

These models gain flexibility at the cost of additional complexity and more parameters to calibrate.

> **Key Concept:** Vasicek and CIR are foundational stepping stones. Mastering them is essential before tackling more advanced models. They illustrate the core mean-reversion concepts that all modern interest rate models inherit.

## 14. References

1. Vasicek, O. "An Equilibrium Characterization of the Term Structure," *JFE*, 1977.
2. Cox, J. C., Ingersoll, J. E. & Ross, S. A. "A Theory of the Term Structure," *Econometrica*, 1985.
3. Uhlenbeck, G. E. & Ornstein, L. S. "On the Theory of Brownian Motion," *Physical Review*, 1930.
4. Hull, J. C. *Options, Futures, and Other Derivatives*, 11th ed., Pearson, 2022.
5. Shreve, S. *Stochastic Calculus for Finance II*, Springer, 2004.
6. Brigo, D. & Mercurio, F. *Interest Rate Models*, Springer, 2006.## 14. References

### Foundational papers

* **Vasicek, O. (1977)** — *"An equilibrium characterization of the term structure,"* Journal of Financial Economics. The original short-rate model.

* **Cox, J. C., Ingersoll, J. E., & Ross, S. A. (1985)** — *"A theory of the term structure of interest rates,"* Econometrica. Introduced CIR.

* **Hull, J., & White, A. (1990)** — *"Pricing interest-rate-derivative securities,"* The Review of Financial Studies. Hull-White model.

* **Heath, D., Jarrow, R., & Morton, A. (1992)** — *"Bond pricing and the term structure of interest rates,"* Econometrica. The HJM framework.

### Standard textbooks

* **Brigo, D., & Mercurio, F.** — *Interest Rate Models — Theory and Practice* (2nd ed.). The definitive reference for short-rate and HJM models.
* **Hull, J. C.** — *Options, Futures, and Other Derivatives*. Chapter on interest rate models gives accessible coverage.
* **Tuckman, B., & Serrat, A.** — *Fixed Income Securities*. Practitioner-focused with detailed CIR derivations.

### CFA curriculum readings

CFA Level 1 introduces yield curves and mean reversion conceptually. Level 2 covers Vasicek and CIR with formulas. Level 3 expands to multi-factor models, calibration, and applications in fixed-income portfolio management.